# 🕸️ Usecase 1: "The Table is a Graph" — Trace Inspection & DAG Analysis

Based on the Google Cloud technical blog [*Your BigQuery Agent Analytics table is a graph. Here's how to see it via SDK*](https://medium.com/google-cloud/your-bigquery-agent-analytics-table-is-a-graph-heres-how-to-see-it-via-sdk-920b4ea14731), this notebook demonstrates how to transform flat, raw `agent_events` rows in BigQuery into hierarchical, human-readable execution trees and ASCII DAGs using the Python SDK (`bigquery-agent-analytics`).

```mermaid
flowchart LR
    A[(BigQuery agent_events)] -->|parent_span_id & span_id| B[SDK Client Trace Builder]
    B --> C[Turn-by-Turn Execution Tree]
    B --> D[ASCII DAG Render trace.render]
    B --> E[Tool Provenance & Handoff Audit]
```

### Key Derived Metrics & Capabilities:
- **Hierarchical Provenance**: Reconstructing parent-child relationships across `USER_MESSAGE_RECEIVED` -> `AGENT_STARTING` -> `LLM_REQUEST` -> `TOOL_COMPLETED` -> `AGENT_RESPONSE`.
- **Tool Execution Provenance**: Tracing tool origin, execution latency, and return payloads.
- **Handoff Tracking**: Monitoring `AGENT_TRANSFER` events across coordinating agents.

In [1]:
import os
from google.auth import default
from google.cloud import bigquery
from bigquery_agent_analytics import Client

# Standard ADC Authentication for centralized BigQuery warehouse
credentials, _ = default()
PROJECT_ID = "nikunjbhartia-test-clients"
DATASET_ID = "agent_analytics"
TABLE_ID = "agent_events"
BQ_LOCATION = "asia-southeast1"

client = Client(
    project_id=PROJECT_ID,
    dataset_id=DATASET_ID,
    table_id=TABLE_ID,
    location=BQ_LOCATION,
)
bq_client = bigquery.Client(project=PROJECT_ID, credentials=credentials)

print(f"✅ Connected SDK Client to `{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}` ({BQ_LOCATION})")


/Users/nikunjbhartia/Desktop/projects/agents/lineage-agent/.venv/lib/python3.14/site-packages/google/auth/_default.py:113: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


/Users/nikunjbhartia/Desktop/projects/agents/lineage-agent/.venv/lib/python3.14/site-packages/google/auth/_default.py:113: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


✅ Connected SDK Client to `nikunjbhartia-test-clients.agent_analytics.agent_events` (asia-southeast1)


## 1) Search & Filter Recorded Traces Across Team Agents

List all recorded session traces and inspect summary metrics (Agent Name, Trace ID, Session ID, Spans, Total Latency).

In [2]:
traces = client.list_traces()
print(f"Total Recorded Trace Session(s) in `{DATASET_ID}.{TABLE_ID}`: {len(traces)}\n")

for idx, t in enumerate(traces[:15], 1):
    agent_name = getattr(t, "agent_name", "lineage_agent")
    print(f"[{idx}] Agent: {agent_name:<15} | Trace: {t.trace_id} | Session: {t.session_id} | Spans: {len(t.spans)} | Latency: {t.total_latency_ms} ms")


Total Recorded Trace Session(s) in `agent_analytics.agent_events`: 4

[1] Agent: lineage_agent   | Trace: f7ff9bab927e01d63007b6bbffaa7fea | Session: 3660a922-70d0-471d-8bb9-25e00df3035e | Spans: 64 | Latency: 487765.273 ms
[2] Agent: lineage_agent   | Trace: 2f61167751094285b82ad8900e96f074 | Session: test_bq_bq_conversation_analytics_agent_a112fd57 | Spans: 12 | Latency: 11408.602 ms
[3] Agent: lineage_agent   | Trace: 6b47ab72561145eabd17ee422e113002 | Session: poc_session_sample_repo_1 | Spans: 19 | Latency: 82258.137 ms
[4] Agent: lineage_agent   | Trace: 88c9398df2ea47f0b45758ada960e058 | Session: test_session_1 | Spans: 15 | Latency: 51534.745 ms


## 2) Visual ASCII DAG Rendering (`trace.render()`)

Calling `trace.render()` prints the full conversation graph, unnesting timestamps, tool inputs, and error flags.

In [3]:
if traces:
    latest = traces[0]
    print(f"=== HIERARCHICAL EXECUTION DAG: {latest.trace_id} (Session: {latest.session_id}) ===")
    print(latest.render())
else:
    print("No traces available to render.")


=== HIERARCHICAL EXECUTION DAG: f7ff9bab927e01d63007b6bbffaa7fea (Session: 3660a922-70d0-471d-8bb9-25e00df3035e) ===
Trace: f7ff9bab927e01d63007b6bbffaa7fea | Session: 3660a922-70d0-471d-8bb9-25e00df3035e | 487765ms
└─ [✓] USER_MESSAGE_RECEIVED [bq_conversation_analytics_agent] - can you find these conversations in the table bigquery
└─ [✓] INVOCATION_STARTING [bq_conversation_analytics_agent]
└─ [✓] AGENT_RESPONSE [bq_conversation_analytics_agent] - Here are the specific conversations (sessions) currently stored in the `agent_events` table. 

It looks like we have ...
└─ [✓] INVOCATION_COMPLETED [bq_conversation_analytics_agent] (36789ms)
   ├─ [✓] AGENT_STARTING [bq_conversation_analytics_agent] - You are the BigQuery Conversation Analytics Agent, an autonomous AI data scientist and observability specialist built...
   └─ [✓] AGENT_COMPLETED [bq_conversation_analytics_agent] (15668ms)
      ├─ [✓] LLM_REQUEST [bq_conversation_analytics_agent] (gemini-pro-latest) - lisr datasets
     

## 3) Span-Level Provenance & Tool Execution Audit

Let's inspect every span in the trace to audit latency, status, and error messages across agent turns.

In [4]:
if traces:
    latest = traces[0]
    print(f"Total Spans: {len(latest.spans)}\n")
    for idx, span in enumerate(latest.spans, 1):
        st = getattr(span, "span_type", "UNKNOWN")
        name = getattr(span, "name", "N/A")
        dur = getattr(span, "duration_ms", 0)
        err = getattr(span, "error", None)
        status = "❌ ERROR" if err or "ERROR" in st else "✅ OK"
        print(f"  {idx:02d}. [{status}] {st:<22} | {name:<25} | {dur:>6} ms")
        if err:
            print(f"      -> Error detail: {err}")


Total Spans: 64

  01. [✅ OK] UNKNOWN                | N/A                       |      0 ms
  02. [✅ OK] UNKNOWN                | N/A                       |      0 ms
  03. [✅ OK] UNKNOWN                | N/A                       |      0 ms
  04. [✅ OK] UNKNOWN                | N/A                       |      0 ms
  05. [✅ OK] UNKNOWN                | N/A                       |      0 ms
  06. [✅ OK] UNKNOWN                | N/A                       |      0 ms
  07. [✅ OK] UNKNOWN                | N/A                       |      0 ms
  08. [✅ OK] UNKNOWN                | N/A                       |      0 ms
  09. [✅ OK] UNKNOWN                | N/A                       |      0 ms
  10. [✅ OK] UNKNOWN                | N/A                       |      0 ms
  11. [✅ OK] UNKNOWN                | N/A                       |      0 ms
  12. [✅ OK] UNKNOWN                | N/A                       |      0 ms
  13. [✅ OK] UNKNOWN                | N/A                       |      